# Teste do kit de edição de vídeo no Colab (GPU grátis)

1. **Ambiente de execução > Alterar o tipo de ambiente > GPU T4**.
2. **Ambiente de execução > Executar tudo**. Não precisa do Drive.
3. No fim, copie as linhas que começam com `RESULTADO` e cole para o Claude.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo SEM GPU
!mkdir -p /content/kit

In [ ]:
%%writefile /content/kit/ambiente.py
# -*- coding: utf-8 -*-
"""Conhece o computador (ou o Colab) antes de editar. Roda uma vez e guarda em cache.

Uso:
    python ambiente.py            # usa o cache se tiver menos de 30 dias
    python ambiente.py --refazer  # mede de novo (ferramenta instalada, erro estranho, outro PC)

Grava ~/.edicao-video/ambiente.json (completo) e imprime um resumo curto, que é o
que o Claude deve ler (economia de tokens). Também cria/atualiza
~/.edicao-video/config.json SÓ com as chaves que ele conhece (skill, scripts,
modelos, drive); chave que o usuário pôs à mão não é apagada nem trocada.

O codificador de vídeo é testado de verdade (0,5 s de vídeo): placa listada pelo
FFmpeg não quer dizer que funciona (ex.: GeForce MX150 lista NVENC e não tem).
"""
import json
import os
import platform
import shutil
import string
import subprocess
import sys
import time

DADOS = os.path.join(os.path.expanduser("~"), ".edicao-video")
ARQ_AMB = os.path.join(DADOS, "ambiente.json")
ARQ_CFG = os.path.join(DADOS, "config.json")
SCRIPTS = os.path.dirname(os.path.abspath(__file__))
SKILL = os.path.dirname(SCRIPTS)
VALIDADE_DIAS = 30
MODELO_GGML = "ggml-small.bin"
# ordem de preferência; libx264 é a garantia
CODIFICADORES = ["h264_nvenc", "h264_videotoolbox", "libx264"]


def _roda(cmd, timeout=60):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace",
                           timeout=timeout)
        return r.returncode, r.stdout + r.stderr
    except (OSError, subprocess.TimeoutExpired) as e:
        return -1, str(e)


def le_json(arq):
    try:
        with open(arq, encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}


def grava_json(arq, dados):
    os.makedirs(os.path.dirname(arq), exist_ok=True)
    tmp = arq + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=1)
    os.replace(tmp, arq)


def config():
    return le_json(ARQ_CFG)


def atualiza_config(novos):
    """Merge por chave presente: só escreve o que veio em `novos`."""
    cfg = config()
    cfg.update({k: v for k, v in novos.items() if v is not None})
    grava_json(ARQ_CFG, cfg)
    return cfg


def pasta_modelos():
    return (os.environ.get("WHISPER_MODELOS") or config().get("modelos")
            or os.path.join(DADOS, "modelos"))


def em_colab():
    return "COLAB_RELEASE_TAG" in os.environ or os.path.isdir("/content/sample_data")


def ram_gb():
    try:
        if os.name == "nt":
            import ctypes

            class M(ctypes.Structure):
                _fields_ = [("l", ctypes.c_ulong), ("c", ctypes.c_ulong), ("t", ctypes.c_ulonglong),
                            ("a", ctypes.c_ulonglong), ("x", ctypes.c_ulonglong * 5)]
            m = M()
            m.l = ctypes.sizeof(M)
            ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(m))
            return round(m.t / 2 ** 30, 1)
        if sys.platform == "darwin":
            return round(int(_roda(["sysctl", "-n", "hw.memsize"])[1].strip()) / 2 ** 30, 1)
        with open("/proc/meminfo") as f:
            return round(int(f.readline().split()[1]) / 2 ** 20, 1)
    except Exception:
        return None


def acha_drive():
    """Pasta raiz do Google Drive sincronizado (Drive para computador) ou montado no Colab."""
    cands = []
    if os.path.isdir("/content/drive/MyDrive"):
        cands.append("/content/drive/MyDrive")
    if os.name == "nt":
        for L in string.ascii_uppercase[3:]:
            cands += [f"{L}:\\Meu Drive", f"{L}:\\My Drive"]
    home = os.path.expanduser("~")
    cs = os.path.join(home, "Library", "CloudStorage")
    if os.path.isdir(cs):
        for d in os.listdir(cs):
            if d.startswith("GoogleDrive"):
                cands += [os.path.join(cs, d, "Meu Drive"), os.path.join(cs, d, "My Drive")]
    cands += [os.path.join(home, "Google Drive"), os.path.join(home, "Meu Drive")]
    for c in cands:
        if os.path.isdir(c):
            return c
    return None


def gpu_nvidia():
    if not shutil.which("nvidia-smi"):
        return None
    c, s = _roda(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], 20)
    return s.strip().splitlines()[0] if c == 0 and s.strip() else None


def faster_whisper():
    try:
        import importlib.util
        if importlib.util.find_spec("faster_whisper") is None:
            return {"instalado": False, "cuda": False}
        import ctranslate2
        return {"instalado": True, "cuda": ctranslate2.get_cuda_device_count() > 0}
    except Exception:
        return {"instalado": False, "cuda": False}


def ffmpeg_info():
    exe = shutil.which("ffmpeg")
    if not exe:
        return {"caminho": None}
    _, v = _roda(["ffmpeg", "-hide_banner", "-version"])
    _, fl = _roda(["ffmpeg", "-hide_banner", "-filters"])
    nomes = {ln.split()[1] for ln in fl.splitlines() if len(ln.split()) > 2 and ln.startswith(" ")}
    filtros = {f: f in nomes for f in ("whisper", "subtitles", "ass", "zscale", "ebur128", "loudnorm",
                                        "afftdn", "xfade", "vidstabdetect")}
    cods = {}
    for c in CODIFICADORES:
        rc, _ = _roda(["ffmpeg", "-hide_banner", "-v", "error", "-f", "lavfi", "-i",
                       "color=c=gray:s=1080x1920:r=30:d=0.5", "-c:v", c, "-pix_fmt", "yuv420p",
                       "-f", "null", "-"], 60)
        cods[c] = rc == 0
    return {"caminho": exe, "versao": v.splitlines()[0] if v else "", "filtros": filtros,
            "codificadores_ok": cods,
            "codificador": next((c for c in CODIFICADORES if cods.get(c)), None)}


def disco_livre_gb(pasta):
    try:
        return round(shutil.disk_usage(pasta).free / 2 ** 30, 1)
    except OSError:
        return None


def libs_python():
    out = {}
    for m in ("PIL", "numpy"):
        try:
            __import__(m)
            out[m] = True
        except ImportError:
            out[m] = False
    return out


def veredito(a):
    falta = []
    if not all(a["python_libs"].values()):
        falta.append("pip install pillow numpy")
    ff = a["ffmpeg"]
    if not ff.get("caminho"):
        falta.append("FFmpeg")
    else:
        for f in ("subtitles", "zscale", "ebur128"):
            if not ff["filtros"].get(f):
                falta.append(f"FFmpeg sem o filtro {f} (instalar build completo)")
    tr = a["transcricao"]
    if not tr["motor"]:
        falta.append("transcrição: baixar modelo (baixar_modelo.py) ou pip install faster-whisper")
    obs = []
    if a["ram_gb"] and a["ram_gb"] < 8:
        obs.append(f"pouca memória ({a['ram_gb']} GB)")
    if a["disco_livre_gb"] is not None and a["disco_livre_gb"] < 10:
        obs.append(f"pouco disco livre ({a['disco_livre_gb']} GB)")
    t = a.get("teste_velocidade") or {}
    fator = t.get("fator_tempo_real")
    lento = fator is not None and fator < 0.25
    if lento:
        obs.append(f"lento: {fator}x tempo real")
    viavel = not falta and not obs
    if falta:
        rec = "instalar o que falta"
    elif lento or (a["ram_gb"] and a["ram_gb"] < 8):
        rec = "usar o Colab (GPU grátis) para transcrever e gravar o vídeo final"
    elif fator is None:
        rec = "rodar teste_velocidade.py para decidir entre local e Colab"
    else:
        rec = "editar neste computador"
    return {"falta": falta, "observacoes": obs, "local_viavel": viavel, "recomendacao": rec}


def mede():
    fw = faster_whisper()
    modelos = pasta_modelos()
    ff = ffmpeg_info()
    ggml = os.path.join(modelos, MODELO_GGML)
    if fw["instalado"] and fw["cuda"]:
        motor = "faster-whisper (GPU)"
    elif ff.get("filtros", {}).get("whisper") and os.path.exists(ggml):
        motor = "ffmpeg-whisper (CPU)"
    elif fw["instalado"]:
        motor = "faster-whisper (CPU)"
    else:
        motor = None
    drive = acha_drive()
    anterior = le_json(ARQ_AMB)
    a = {
        "medido_em": time.strftime("%Y-%m-%d %H:%M"),
        "sistema": f"{platform.system()} {platform.release()}",
        "colab": em_colab(),
        "cpu_nucleos": os.cpu_count(),
        "ram_gb": ram_gb(),
        "disco_livre_gb": disco_livre_gb(drive or os.path.expanduser("~")),
        "python": platform.python_version(),
        "python_libs": libs_python(),
        "gpu_nvidia": gpu_nvidia(),
        "ffmpeg": ff,
        "transcricao": {"motor": motor, "faster_whisper": fw, "modelo_ggml": ggml if os.path.exists(ggml) else None},
        "drive": drive,
        "teste_velocidade": anterior.get("teste_velocidade") if anterior.get("sistema") == f"{platform.system()} {platform.release()}" else None,
    }
    a["veredito"] = veredito(a)
    grava_json(ARQ_AMB, a)
    atualiza_config({"skill": SKILL, "scripts": SCRIPTS, "modelos": modelos, "drive": drive})
    return a


def carrega(refazer=False):
    a = le_json(ARQ_AMB)
    if refazer or not a or not a.get("medido_em"):
        return mede()
    idade = (time.time() - time.mktime(time.strptime(a["medido_em"], "%Y-%m-%d %H:%M"))) / 86400
    return mede() if idade > VALIDADE_DIAS else a


def codificador_args(qualidade_cpu=("-crf", "17", "-preset", "medium")):
    """Argumentos -c:v para o vídeo final, conforme o que funcionou neste ambiente.
    EDICAO_VIDEO_ENCODER força um (ex.: libx264)."""
    c = os.environ.get("EDICAO_VIDEO_ENCODER") or le_json(ARQ_AMB).get("ffmpeg", {}).get("codificador") or "libx264"
    if c == "h264_nvenc":
        return ["-c:v", "h264_nvenc", "-preset", "p6", "-tune", "hq", "-rc", "vbr", "-cq", "19",
                "-b:v", "0", "-profile:v", "high"]
    if c == "h264_videotoolbox":
        return ["-c:v", "h264_videotoolbox", "-q:v", "70", "-profile:v", "high"]
    return ["-c:v", "libx264", *qualidade_cpu]


def resumo(a):
    ff, tr, v = a["ffmpeg"], a["transcricao"], a["veredito"]
    linhas = [
        f"Ambiente ({a['medido_em']}): {a['sistema']}{' [Colab]' if a['colab'] else ''}, "
        f"{a['cpu_nucleos']} núcleos, {a['ram_gb']} GB RAM, {a['disco_livre_gb']} GB livres",
        f"GPU NVIDIA: {a['gpu_nvidia'] or 'não'} | codificador: {ff.get('codificador') or '-'} | "
        f"transcrição: {tr['motor'] or 'nenhuma'}",
        f"FFmpeg: {ff.get('versao', 'não encontrado')[:60]}",
        f"Google Drive: {a['drive'] or 'não encontrado neste computador'}",
    ]
    t = a.get("teste_velocidade")
    if t:
        linhas.append(f"Velocidade: {t['fator_tempo_real']}x tempo real ({t['codificador']}); "
                      f"Reels de 60 s ~ {t['estimativa_60s_min']} min")
    linhas.append(f"Falta: {'; '.join(v['falta']) or 'nada'} | Obs.: {'; '.join(v['observacoes']) or '-'}")
    linhas.append(f"Recomendação: {v['recomendacao']}")
    return "\n".join(linhas)


if __name__ == "__main__":
    sys.stdout.reconfigure(encoding="utf-8")
    print(resumo(carrega("--refazer" in sys.argv)))


In [ ]:
%%writefile /content/kit/teste_velocidade.py
# -*- coding: utf-8 -*-
"""Mede quanto tempo este computador leva para gravar um Reels, sem usar vídeo de ninguém.

Uso: python teste_velocidade.py [segundos=8]

Gera um vídeo sintético 1080x1920 (imagem de teste + tom), aplica o mesmo tipo de
filtro da montagem (escala lanczos, punch-in, legenda ASS queimada) e grava com o
codificador escolhido pelo ambiente.py. Resultado: "fator de tempo real"
(1.0 = 1 s de vídeo gravado em 1 s) e estimativa para um Reels de 60 s.
Salva em ~/.edicao-video/ambiente.json (chave teste_velocidade). Leve: poucos
segundos de processamento, tudo em pasta temporária apagada no fim.
"""
import os
import shutil
import subprocess
import sys
import tempfile
import time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ambiente  # noqa: E402

ASS = """[Script Info]
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Leg,Arial,80,&H004DE1FF,&H00FFFFFF,&H00000000,&H80000000,1,0,0,0,100,100,0,0,1,7,3,2,40,40,672,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
Dialogue: 0,0:00:00.00,0:00:30.00,Leg,,0,0,0,,{\\k40}TESTE {\\k40}DE {\\k40}VELOCIDADE
"""


def main():
    seg = float(sys.argv[1]) if len(sys.argv) > 1 else 8.0
    a = ambiente.carrega()
    cod = ambiente.codificador_args()
    tmp = tempfile.mkdtemp(prefix="ev_teste_")
    try:
        ass = os.path.join(tmp, "t.ass")
        with open(ass, "w", encoding="utf-8") as f:
            f.write(ASS)
        cwd_ass = "t.ass"   # roda dentro da pasta: evita escapar C: no filtro
        vf = ("format=yuv420p,scale=1080:1920:flags=lanczos,crop=914:1626:80:120,"
              f"scale=1080:1920:flags=lanczos,setsar=1,subtitles={cwd_ass}")
        cmd = ["ffmpeg", "-hide_banner", "-v", "error", "-y",
               "-f", "lavfi", "-i", f"testsrc2=s=1080x1920:r=30:d={seg}",
               "-f", "lavfi", "-i", f"sine=f=440:d={seg}:sample_rate=48000",
               "-vf", vf, *cod, "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k",
               "-movflags", "+faststart", "saida.mp4"]
        t0 = time.time()
        r = subprocess.run(cmd, cwd=tmp, capture_output=True, text=True, encoding="utf-8", errors="replace")
        dt = time.time() - t0
        if r.returncode != 0:
            sys.exit("teste falhou:\n" + r.stderr[-1500:])
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    fator = round(seg / dt, 2)
    res = {"medido_em": time.strftime("%Y-%m-%d %H:%M"), "segundos_video": seg,
           "segundos_gastos": round(dt, 1), "fator_tempo_real": fator, "codificador": cod[1],
           "estimativa_60s_min": round(60 / fator / 60 * 1.6, 1)}   # 1,6 = tonemap, áudio e 2ª passada de loudness
    a["teste_velocidade"] = res
    a["veredito"] = ambiente.veredito(a)
    ambiente.grava_json(ambiente.ARQ_AMB, a)
    print(ambiente.resumo(a))


if __name__ == "__main__":
    sys.stdout.reconfigure(encoding="utf-8")
    main()


In [ ]:
%%writefile /content/kit/whisper.py
# -*- coding: utf-8 -*-
"""Transcreve fala em SRT.

Uso: python whisper.py <entrada> <saida.srt> [ini] [fim] [max_len]
  max_len=1 (padrão) = uma palavra por legenda, com tempo por palavra.
  ini/fim em segundos: retranscrever uma janela isolada para conferir trecho duvidoso
  (os tempos do SRT saem relativos a ini).

Motor (WHISPER_MOTOR = auto | ffmpeg | faster), auto escolhe nesta ordem:
  1. faster-whisper na GPU NVIDIA (Colab): rápido, modelo WHISPER_FASTER_MODELO (padrão medium);
  2. filtro whisper do FFmpeg (8.0+) com o modelo ggml (baixar_modelo.py);
  3. faster-whisper na CPU (modelo small).
Idioma: WHISPER_LANG (padrão pt). Pasta dos modelos: ~/.edicao-video/modelos
(ou WHISPER_MODELOS / config.json). O filtro do FFmpeg não aceita 'C:' no caminho
do modelo: roda com a pasta do modelo como diretório atual.
"""
import os
import shutil
import subprocess
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ambiente  # noqa: E402

LINGUA = os.environ.get("WHISPER_LANG", "pt")
MODELO_GGML = os.environ.get("WHISPER_MODELO", ambiente.MODELO_GGML)


def _srt_ts(t):
    ms = int(round(max(0.0, t) * 1000))
    return f"{ms // 3600000:02d}:{ms // 60000 % 60:02d}:{ms // 1000 % 60:02d},{ms % 1000:03d}"


def _cuda():
    return ambiente.faster_whisper()["cuda"]


def _carrega_cuda_pip():
    """Carrega cuBLAS/cuDNN dos pacotes pip nvidia-* (o Colab nem sempre traz as versões que o
    ctranslate2 pede: 'libcublas.so.12 is not found', 03/11/2026). Sem os pacotes, não faz nada."""
    if not sys.platform.startswith("linux"):
        return
    import ctypes
    import glob
    import site
    bases = site.getsitepackages() + [site.getusersitepackages()]
    libs = []
    for b in bases:
        for p in ("nvidia/cublas/lib/libcublasLt.so.*", "nvidia/cublas/lib/libcublas.so.*",
                  "nvidia/cudnn/lib/libcudnn*.so.*"):
            libs += sorted(glob.glob(os.path.join(b, p)))
    for _ in range(2):          # 2ª volta pega as que dependiam de outra ainda não carregada
        for so in libs:
            try:
                ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass


def escolhe_motor():
    m = os.environ.get("WHISPER_MOTOR", "auto")
    if m != "auto":
        return m
    fw = ambiente.faster_whisper()
    if fw["instalado"] and fw["cuda"]:
        return "faster"
    if os.path.exists(os.path.join(ambiente.pasta_modelos(), MODELO_GGML)) and shutil.which("ffmpeg"):
        return "ffmpeg"
    if fw["instalado"]:
        return "faster"
    sys.exit("sem motor de transcrição: rodar baixar_modelo.py (FFmpeg 8+) ou pip install faster-whisper")


def via_ffmpeg(entrada, saida, ini, fim, max_len):
    dest = saida.replace("\\", "/").replace(":", "\\\\:")
    cmd = ["ffmpeg", "-hide_banner", "-v", "error", "-y"]
    if ini:
        cmd += ["-ss", str(ini)]
    if fim:
        cmd += ["-to", str(fim)]
    cmd += ["-i", entrada, "-af",
            f"whisper=model={MODELO_GGML}:language={LINGUA}:queue=600:max_len={max_len}:"
            f"destination={dest}:format=srt", "-f", "null", "-"]
    r = subprocess.run(cmd, cwd=ambiente.pasta_modelos(), capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    if r.returncode != 0 or not os.path.exists(saida):
        sys.exit("whisper (ffmpeg) falhou:\n" + r.stderr[-1500:])


def via_faster(entrada, saida, ini, fim, max_len):
    from faster_whisper import WhisperModel
    gpu = _cuda()
    nome = os.environ.get("WHISPER_FASTER_MODELO", "medium" if gpu else "small")
    import numpy as np
    # áudio decodificado pelo FFmpeg e entregue como array: não depende do PyAV
    # (faster-whisper 1.x + PyAV novo quebra em av.open(metadata_errors=...), Colab 03/11/2026)
    cmd = ["ffmpeg", "-hide_banner", "-v", "error"]
    if ini:
        cmd += ["-ss", str(ini)]
    if fim:
        cmd += ["-to", str(fim)]
    r = subprocess.run(cmd + ["-i", entrada, "-ac", "1", "-ar", "16000", "-f", "f32le", "-"],
                       capture_output=True)
    if r.returncode != 0:
        sys.exit("ffmpeg não leu o áudio:\n" + r.stderr.decode("utf-8", "replace")[-1500:])
    audio = np.frombuffer(r.stdout, dtype=np.float32)
    raiz = os.path.join(ambiente.pasta_modelos(), "faster")

    def roda(gpu, nome):
        modelo = WhisperModel(nome, device="cuda" if gpu else "cpu",
                              compute_type="float16" if gpu else "int8", download_root=raiz)
        segs, _ = modelo.transcribe(audio, language=LINGUA, word_timestamps=True, beam_size=5)
        out = []
        for s in segs:           # a GPU só é usada de fato aqui (gerador)
            if str(max_len) == "1":
                out += [(w.start, w.end, w.word.strip()) for w in (s.words or []) if w.word.strip()]
            else:
                out.append((s.start, s.end, s.text.strip()))
        return out

    if gpu:
        _carrega_cuda_pip()
        try:
            itens = roda(True, nome)
        except RuntimeError as e:
            print(f"AVISO: GPU falhou ({str(e)[:120]}); transcrevendo na CPU (mais lento).", file=sys.stderr)
            itens = roda(False, os.environ.get("WHISPER_FASTER_MODELO", "small"))
    else:
        itens = roda(False, nome)
    with open(saida, "w", encoding="utf-8", newline="\n") as f:
        for i, (a, b, t) in enumerate(itens, 1):
            f.write(f"{i}\n{_srt_ts(a)} --> {_srt_ts(b)}\n{t}\n\n")


def transcreve(entrada, saida, ini=None, fim=None, max_len="1"):
    entrada, saida = os.path.abspath(entrada), os.path.abspath(saida)
    os.makedirs(os.path.dirname(saida), exist_ok=True)
    motor = escolhe_motor()
    (via_faster if motor == "faster" else via_ffmpeg)(entrada, saida, ini, fim, max_len)
    return saida


if __name__ == "__main__":
    sys.stdout.reconfigure(encoding="utf-8")
    if len(sys.argv) < 3:
        sys.exit(__doc__)
    a = sys.argv[1:] + [None] * 3
    out = transcreve(a[0], a[1], a[2], a[3], a[4] or "1")
    print(open(out, encoding="utf-8").read())


In [ ]:
%%writefile /content/kit/colab_worker.py
# -*- coding: utf-8 -*-
"""Roda NO COLAB (GPU grátis). Prepara o ambiente e executa as tarefas que o Claude
deixou na pasta do Google Drive. O usuário só abre o notebook e clica em "Executar tudo".

Uso (célula do notebook, depois de montar o Drive):
    python colab_worker.py [--esperar 20] [--uma-vez] [--teste]
      --esperar N : fica N minutos esperando tarefa nova antes de parar (padrão 20)
      --uma-vez   : executa o que estiver na fila e para
      --teste     : roda teste_velocidade.py e grava o resultado no Drive

Fila no Drive (criada pelo colab_job.py no computador do usuário):
    Edicao-Video/_colab/fila/<id>.json   -> {"id", "passos": [{"cmd": [...], "cwd": "..."}]}
    Edicao-Video/_colab/feito/<id>.json  <- resultado de cada passo (código, tempo, fim da saída)
Marcadores nos comandos: {drive} = raiz do Meu Drive, {scripts} = esta pasta.
Só executa python, ffmpeg e ffprobe; script python tem de estar dentro do Drive.
"""
import glob
import json
import os
import shutil
import subprocess
import sys
import time

DRIVE = "/content/drive/MyDrive"
BASE = os.path.join(DRIVE, "Edicao-Video")
FILA = os.path.join(BASE, "_colab", "fila")
FEITO = os.path.join(BASE, "_colab", "feito")
SCRIPTS = os.path.dirname(os.path.abspath(__file__))
FFMPEG_URL = ("https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/"
              "ffmpeg-master-latest-linux64-gpl.tar.xz")
PERMITIDOS = {"python", "python3", "ffmpeg", "ffprobe"}


def sh(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace", **kw)


def ffmpeg_ok():
    if not shutil.which("ffmpeg"):
        return False
    fl = sh(["ffmpeg", "-hide_banner", "-filters"]).stdout
    if not all(f" {f} " in fl for f in ("subtitles", "zscale", "ebur128")):
        return False
    r = sh(["ffmpeg", "-v", "error", "-f", "lavfi", "-i", "color=s=1080x1920:d=0.5", "-c:v", "h264_nvenc",
            "-f", "null", "-"])
    return r.returncode == 0


def prepara():
    t0 = time.time()
    if not os.path.isdir(DRIVE):
        sys.exit("Drive não montado. Rode antes a célula: from google.colab import drive; drive.mount('/content/drive')")
    if sh(["nvidia-smi"]).returncode != 0:
        print("AVISO: sem GPU. Menu Ambiente de execução > Alterar o tipo > GPU T4, e rodar de novo.")
    try:
        import faster_whisper  # noqa: F401
    except ImportError:
        print("instalando faster-whisper...", flush=True)
        sh([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"])
    # cuBLAS/cuDNN que o ctranslate2 pede (o Colab pode não ter a versão certa)
    sh([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"])
    if not ffmpeg_ok():
        print("baixando FFmpeg completo (BtbN)...", flush=True)
        os.makedirs("/content/ffmpeg", exist_ok=True)
        sh(["bash", "-c", f"curl -sL {FFMPEG_URL} | tar -xJ -C /content/ffmpeg --strip-components=1"])
        os.environ["PATH"] = "/content/ffmpeg/bin:" + os.environ["PATH"]
    for d in (FILA, FEITO):
        os.makedirs(d, exist_ok=True)
    os.environ["EDICAO_VIDEO_SCRIPTS"] = SCRIPTS
    r = sh([sys.executable, os.path.join(SCRIPTS, "ambiente.py"), "--refazer"])
    print(r.stdout.strip() or r.stderr[-800:])
    print(f"preparado em {time.time() - t0:.0f} s", flush=True)


def troca(v):
    return v.replace("{drive}", DRIVE).replace("{scripts}", SCRIPTS) if isinstance(v, str) else v


def valida(cmd):
    exe = os.path.basename(cmd[0])
    if exe not in PERMITIDOS:
        return f"comando não permitido: {exe}"
    if exe.startswith("python"):
        alvo = os.path.realpath(cmd[1]) if len(cmd) > 1 else ""
        if not (alvo.startswith(os.path.realpath(DRIVE)) or alvo.startswith(os.path.realpath(SCRIPTS))):
            return f"script fora do Drive: {cmd[1] if len(cmd) > 1 else ''}"
    return None


def executa(arq):
    with open(arq, encoding="utf-8") as f:
        job = json.load(f)
    jid = job.get("id") or os.path.splitext(os.path.basename(arq))[0]
    print(f"\n== tarefa {jid}", flush=True)
    res = {"id": jid, "inicio": time.strftime("%Y-%m-%d %H:%M:%S"), "passos": [], "ok": True}
    for p in job.get("passos", []):
        cmd = [troca(c) for c in p["cmd"]]
        erro = valida(cmd)
        if not erro and cmd[0] in ("python", "python3"):
            cmd[0] = sys.executable
        if erro:
            res["passos"].append({"cmd": p["cmd"], "codigo": -1, "saida": erro})
            res["ok"] = False
            break
        t0 = time.time()
        r = sh(cmd, cwd=troca(p.get("cwd")) or None)
        saida = (r.stdout + r.stderr)[-2000:]
        res["passos"].append({"cmd": p["cmd"], "codigo": r.returncode, "segundos": round(time.time() - t0, 1),
                              "saida": saida})
        print(f"  {os.path.basename(cmd[1] if len(cmd) > 1 else cmd[0])}: código {r.returncode}, "
              f"{time.time() - t0:.0f} s", flush=True)
        if r.returncode != 0:
            res["ok"] = False
            break
    res["fim"] = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(os.path.join(FEITO, f"{jid}.json"), "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False, indent=1)
    shutil.move(arq, os.path.join(FEITO, f"{jid}.pedido.json"))
    print("  OK" if res["ok"] else "  FALHOU (detalhe em _colab/feito)", flush=True)


def main():
    a = sys.argv[1:]
    esperar = float(a[a.index("--esperar") + 1]) if "--esperar" in a else 20.0
    prepara()
    if "--teste" in a:
        r = sh([sys.executable, os.path.join(SCRIPTS, "teste_velocidade.py"), "8"])
        print(r.stdout.strip() or r.stderr[-800:])
        with open(os.path.join(FEITO, "teste_colab.txt"), "w", encoding="utf-8") as f:
            f.write(r.stdout + r.stderr)
    ultimo = time.time()
    while True:
        jobs = sorted(glob.glob(os.path.join(FILA, "*.json")))
        for j in jobs:
            executa(j)
            ultimo = time.time()
        if "--uma-vez" in a or time.time() - ultimo > esperar * 60:
            print("\nFila vazia. Pode fechar o Colab (ou rodar a célula de novo quando houver tarefa).")
            return
        time.sleep(15)


if __name__ == "__main__":
    main()


In [ ]:
import os, sys, subprocess, time
sys.path.insert(0, '/content/kit')
import colab_worker as w
t0 = time.time()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faster-whisper', 'nvidia-cublas-cu12', 'nvidia-cudnn-cu12==9.*'])
if not w.ffmpeg_ok():
    print('baixando FFmpeg completo...')
    os.makedirs('/content/ffmpeg', exist_ok=True)
    subprocess.run(['bash', '-c', f'curl -sL {w.FFMPEG_URL} | tar -xJ -C /content/ffmpeg --strip-components=1'])
    os.environ['PATH'] = '/content/ffmpeg/bin:' + os.environ['PATH']
print(f'RESULTADO preparo: {time.time()-t0:.0f} s, ffmpeg completo com NVENC: {w.ffmpeg_ok()}')

In [ ]:
!python /content/kit/ambiente.py --refazer
!python /content/kit/teste_velocidade.py 8 | sed 's/^/RESULTADO /'
!EDICAO_VIDEO_ENCODER=libx264 python /content/kit/teste_velocidade.py 8 | grep Velocidade | sed 's/^/RESULTADO CPU-do-Colab /'

In [ ]:
# transcrição na GPU: fala sintética em inglês só para medir velocidade e confirmar CUDA
import subprocess, time
subprocess.run(['ffmpeg','-v','error','-y','-f','lavfi','-i','flite=text=testing one two three:voice=slt','-f','lavfi','-i','anullsrc=r=16000:cl=mono','-filter_complex','[0][1]concat=n=2:v=0:a=1,atrim=0:60','/content/fala.wav'])
import os
if not os.path.exists('/content/fala.wav'):
    subprocess.run(['ffmpeg','-v','error','-y','-f','lavfi','-i','sine=f=300:d=60','/content/fala.wav'])
t0 = time.time()
r = subprocess.run(['python','/content/kit/whisper.py','/content/fala.wav','/content/fala.srt'], capture_output=True, text=True, env={**os.environ, 'WHISPER_LANG': 'en'})
print(r.stdout[:300], r.stderr[-500:])
print(f'RESULTADO transcrição de 60 s de áudio (1ª vez inclui baixar o modelo): {time.time()-t0:.0f} s')